In [2]:
import os
import gradio as gr

from dotenv import load_dotenv

from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.llms import ChatMessage
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.readers.file import PyMuPDFReader

# ==================================================
# 1. Load environment variables
# ==================================================
load_dotenv()

# ==================================================
# 2. LLM + Embedding setup
# ==================================================
Settings.llm = Groq(model="llama-3.3-70b-versatile")
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# ==================================================
# 3. Path to your fixed PDF source
# ==================================================
# Put your PDF file in the same folder as this script,
# or give the full path to the file.
PDF_PATH = "./data.pdf"

# ==================================================
# 4. Global variable for the chatbot
# ==================================================
rag_bot = None

# ==================================================
# 5. Theme and CSS
# ==================================================
custom_theme = gr.themes.Soft(
    primary_hue="green",
    secondary_hue="blue",
    neutral_hue="gray",
    font=gr.themes.GoogleFont("Inter")
)

custom_css = """
.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
    padding: 20px !important;
}
.chatbot-container {
    border-radius: 15px;
    overflow: hidden;
    box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
}
"""

# ==================================================
# 6. Load the fixed PDF and create the RAG bot
# ==================================================
def create_bot_from_fixed_pdf(top_k_value):
    global rag_bot

    if not os.path.exists(PDF_PATH):
        return f"Error: PDF file not found at '{PDF_PATH}'."

    try:
        loader = PyMuPDFReader()
        documents = loader.load_data(file_path=PDF_PATH)

        readable_docs = [
            doc for doc in documents
            if hasattr(doc, "text") and doc.text and doc.text.strip()
        ]

        if not readable_docs:
            return "The PDF was loaded, but no readable text was found."

        index = VectorStoreIndex.from_documents(readable_docs)

        rag_bot = index.as_chat_engine(
            chat_mode="context",
            similarity_top_k=int(top_k_value),
            verbose=True
        )

        total_chars = sum(len(doc.text) for doc in readable_docs)

        return (
            f"Source PDF loaded successfully. "
            f"Loaded {len(readable_docs)} document(s) with about {total_chars} characters."
        )

    except Exception as e:
        return f"Error while loading PDF: {e}"

# ==================================================
# 7. Chat callback
# ==================================================
def custom_rag_bot_callback(message, history, top_k_value):
    global rag_bot

    history = history or []

    if not message.strip():
        return history

    if rag_bot is None:
        history.append({"role": "user", "content": message})
        history.append({
            "role": "assistant",
            "content": "The source PDF is not loaded yet."
        })
        return history

    try:
        rag_bot._retriever._similarity_top_k = int(top_k_value)

        bot_history = []
        for m in history:
            if isinstance(m, dict) and "role" in m and "content" in m:
                bot_history.append(
                    ChatMessage(
                        role=m["role"],
                        content=str(m["content"])
                    )
                )

        response = rag_bot.chat(message, chat_history=bot_history)

        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": response.response})

        return history

    except Exception as e:
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": f"Error: {e}"})
        return history

# ==================================================
# 8. Clear chat
# ==================================================
def clear_chat():
    return [
        {
            "role": "assistant",
            "content": "Hello! Ask me something about the source PDF."
        }
    ], ""

# ==================================================
# 9. Build UI
# ==================================================
with gr.Blocks(theme=custom_theme, css=custom_css, title="Custom RAG Bot UI") as demo_custom:
    gr.HTML("""
    <div style="text-align: center; margin-bottom: 20px;">
        <h1>🤖 RAG Bot</h1>
        <p>This chatbot answers questions from a fixed PDF source.</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1, min_width=260):
            gr.Markdown("## ⚙️ RAG Settings")

            top_k_slider = gr.Slider(
                minimum=1,
                maximum=5,
                step=1,
                value=2,
                label="Top K"
            )

            clear_btn = gr.Button("Clear Chat")

            status_box = gr.Textbox(
                label="Status",
                interactive=False
            )

        with gr.Column(scale=4, elem_classes="chatbot-container"):
            chatbot = gr.Chatbot(
                label="Climate Change Expert",
                height=500,
                layout="bubble",
                value=[
                    {
                        "role": "assistant",
                        "content": "Hello! Ask me something about the source PDF."
                    }
                ]
            )

            msg_input = gr.Textbox(
                label="Your Question",
                placeholder="Ask me something about the source PDF..."
            )

            msg_input.submit(
                fn=custom_rag_bot_callback,
                inputs=[msg_input, chatbot, top_k_slider],
                outputs=[chatbot]
            ).then(
                fn=lambda: "",
                outputs=[msg_input]
            )

            clear_btn.click(
                fn=clear_chat,
                outputs=[chatbot, msg_input]
            )

    # Load the fixed PDF automatically when the app starts
    demo_custom.load(
        fn=create_bot_from_fixed_pdf,
        inputs=[top_k_slider],
        outputs=[status_box]
    )

# ==================================================
# 10. Launch app
# ==================================================
demo_custom.launch(share=True)

2026-03-31 09:11:29,252 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
C:\Users\sanaz\AppData\Local\Temp\ipykernel_8620\1146635368.py:157: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=custom_theme, css=custom_css, title="Custom RAG Bot UI") as demo_custom:
2026-03-31 09:11:31,488 - INFO - HTTP Request: GET http://127.0.0.1:7865/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-03-31 09:11:31,498 - INFO - HTTP Request: HEAD http://127.0.0.1:7865/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7865


2026-03-31 09:11:32,062 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-03-31 09:11:32,283 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"


* Running on public URL: https://bd1443109fd103c29a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2026-03-31 09:11:34,101 - INFO - HTTP Request: HEAD https://bd1443109fd103c29a.gradio.live "HTTP/1.1 200 OK"
